# Notebook Training 04 - MobileNetV3-Large + Attention Module (SE & CBAM)
## Target Pertemuan 11 November 2026: Model Usulan & Ablation Study

---

### 🎯 Tujuan & Ruang Lingkup Progres (11 November):
1. **Implementasi Model Usulan**: Memperkaya arsitektur ringan `MobileNetV3-Large` dengan modul atensi untuk meningkatkan akurasi klasifikasi 13 topeng tradisional Bali tanpa membebani komputasi.
2. **Ablation Study (Uji Ablasi Sejajar)**:
   - `mobilenetv3_base`: Baseline ringan (MobileNetV3-Large polos tanpa modul tambahan).
   - `mobilenetv3_se_head`: MobileNetV3 + Squeeze-and-Excitation (Channel Attention) pada feature map akhir (960 channel).
   - `mobilenetv3_cbam_head`: MobileNetV3 + CBAM (Channel Gate + Spatial Gate) pada feature map akhir.
   - `mobilenetv3_cbam_late`: MobileNetV3 + CBAM pada 3 bottleneck blocks terakhir (blok 13, 14, 15) untuk menangkap fitur multi-skala.
3. **Integritas Data (Anti-Data Leakage)**: Menggunakan *Identity-Based & pHash Grouping* agar seluruh varian rotasi/crop dari satu topeng fisik yang sama terkunci di satu split (tidak bocor ke data uji).
4. **Visualisasi Mandiri (Inline di Colab)**: Seluruh kurva training, confusion matrix, tabel metrik, dan Grad-CAM langsung tertampil di layar notebook tanpa perlu membuka Google Drive.

In [ ]:
# Sel 1 - Setup Lingkungan & Ekstraksi Dataset
!pip -q install grad-cam imagehash seaborn

import os, zipfile
from pathlib import Path

# Mount Drive jika berjalan di Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Berjalan di lokal/non-Colab.")

ZIP_PATH    = '/content/drive/MyDrive/dataset_reorganized.zip'
EXTRACT_DIR = '/content/BaliMask_Clean'

# Ekstraksi dataset hanya jika belum ada di disk lokal Colab (/content)
if os.path.exists(ZIP_PATH) and not os.path.exists(EXTRACT_DIR):
    print("📦 Mengekstrak dataset dari ZIP Google Drive...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print("✅ Ekstraksi selesai ke local runtime.")
elif os.path.exists(EXTRACT_DIR):
    print("✅ Dataset sudah tersedia di local runtime /content/BaliMask_Clean.")
else:
    print("ℹ️ ZIP tidak ditemukan di Drive, mencari dataset lokal di direktori kerja...")

# Auto-detect struktur folder train/valid/test
def find_data_root(base_dir):
    if not os.path.exists(base_dir):
        return base_dir
    if all(os.path.exists(os.path.join(base_dir, s)) for s in ['train', 'valid', 'test']):
        return base_dir
    for item in os.listdir(base_dir):
        sub = os.path.join(base_dir, item)
        if os.path.isdir(sub) and all(os.path.exists(os.path.join(sub, s)) for s in ['train', 'valid', 'test']):
            return sub
    return base_dir

DATA_ROOT = find_data_root(EXTRACT_DIR if os.path.exists(EXTRACT_DIR) else ".")
print(f"✅ DATA_ROOT aktif: {DATA_ROOT}")

In [ ]:
# Sel 2 - Import Library & Konfigurasi Global
import os, json, time, random, copy, re
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, 
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from IPython.display import display

# Path data
BASE_DIR = Path(DATA_ROOT)
DATA_DIR = BASE_DIR / "train" if (BASE_DIR / "train").exists() else BASE_DIR
REAL_DIR = BASE_DIR / "real_world"

# Output direktori (disimpan lokal di runtime Colab agar cepat & disalin ke Drive sebagai backup)
OUT_ROOT = Path("./outputs_mobilenetv3")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Konfigurasi Reproducibility & Hyperparameter
SPLIT_SEED = 42
SEEDS = [42]                 # Gunakan [42] untuk running cepat di kelas, atau [42, 123, 2024] untuk laporan
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2
LABEL_SMOOTHING = 0.1
USE_CLASS_WEIGHTS = True
USE_AMP = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device aktif:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
# Sel 3 - Scanning Dataset & Split Anti-Bocor (Identity-Based + pHash Grouping)
import imagehash
from sklearn.model_selection import StratifiedGroupKFold

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_CSV = OUT_ROOT / "split_leakage_free.csv"

def extract_base_identity(path_str):
    """Mengekstrak identitas dasar objek foto sebelum augmentasi offline (rotasi, Roboflow hash, dll)."""
    name = Path(path_str).name.lower()
    name = re.sub(r'\.(jpg|jpeg|png|webp|bmp)$', '', name)
    name = re.sub(r'(\._rf\.|\.rf\.|_rf\.)[a-f0-9]+$', '', name)   # hapus hash Roboflow
    name = re.sub(r'(_jpg|_jpeg|_png)$', '', name)
    if name.startswith('resized_'):
        name = name[len('resized_'):]
    name = re.sub(r'(_rotated-?\d+|-rotated-?\d+|rotated-?\d+)', '', name)  # hapus rotasi offline
    name = re.sub(r'(_aug\d+|-aug\d+)', '', name)
    return name.strip('-_ ')

def scan_folder(root):
    root = Path(root)
    classes = sorted([d.name for d in root.iterdir() if d.is_dir() and not d.name.startswith('.')])
    rows = []
    for ci, c in enumerate(classes):
        for p in sorted((root / c).rglob("*")):
            if p.suffix.lower() in IMG_EXT:
                rel = str(p.relative_to(root))
                base_id = extract_base_identity(rel)
                rows.append((rel, c, ci, base_id))
    return classes, pd.DataFrame(rows, columns=["path", "label", "y", "base_id"])

def build_leakage_free_groups(df, root, max_dist=4):
    """Menggabungkan kesamaan Identitas Dasar file DAN pHash visual ke dalam satu Grup unik."""
    n = len(df)
    parent = list(range(n))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]
            a = parent[a]
        return a
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    # 1. Kunci kesamaan berdasarkan Base ID (semua varian rotasi/hash terkunci di grup sama)
    id_map = {}
    for i, bid in enumerate(df.base_id):
        if bid in id_map:
            union(id_map[bid], i)
        else:
            id_map[bid] = i

    # 2. Verifikasi tambahan dengan pHash untuk foto identik yang beda nama
    print("🔍 Menghitung pHash untuk verifikasi duplikat visual...")
    hashes = []
    for p in tqdm(df.path, desc="pHash"):
        with Image.open(Path(root) / p) as im:
            hashes.append(imagehash.phash(im.convert("RGB")).hash.flatten())
    arr = np.array(hashes, dtype=np.uint8)
    for i in range(n - 1):
        d = (arr[i + 1:] != arr[i]).sum(axis=1)
        for j in np.where(d <= max_dist)[0]:
            union(i, i + 1 + int(j))

    return np.array([find(i) for i in range(n)])

def make_split(df, groups, seed=SPLIT_SEED):
    # 64% train / 16% val / 20% test bebas kebocoran
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    trval_idx, test_idx = next(sgkf.split(df, df.y, groups))
    sub = df.iloc[trval_idx]
    sgkf2 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    _, va_rel = next(sgkf2.split(sub, sub.y, groups[trval_idx]))
    
    split = np.full(len(df), "train", dtype=object)
    split[trval_idx[va_rel]] = "val"
    split[test_idx] = "test"
    out = df.copy()
    out["group"] = groups
    out["split"] = split
    return out

if SPLIT_CSV.exists():
    df = pd.read_csv(SPLIT_CSV)
    print("✅ Memuat split bebas bocor yang sudah ada:", SPLIT_CSV)
else:
    CLASSES, df = scan_folder(DATA_DIR)
    assert len(df) > 0, f"Tidak ada gambar di {DATA_DIR}"
    groups = build_leakage_free_groups(df, DATA_DIR)
    print(f"Total gambar: {len(df)} | Objek topeng unik: {len(set(groups))} | "
          f"Gambar hasil varian offline: {len(df) - len(set(groups))}")
    df = make_split(df, groups)
    df.to_csv(SPLIT_CSV, index=False)
    print("✅ Split anti-leakage berhasil dibuat & disimpan ke:", SPLIT_CSV)

CLASSES = df.drop_duplicates("y").sort_values("y").label.tolist()
NUM_CLASSES = len(CLASSES)
assert df.groupby("group").split.nunique().max() == 1, "❌ ERROR: Ada varian topeng yang bocor antar-split!"
print(f"\n🛡️ VERIFIKASI SPLIT: 100% BEBAS KEBOCORAN (Max group across split = 1)")
print(f"Total {NUM_CLASSES} kelas topeng:", CLASSES)

# TAMPILKAN TABEL DISTRIBUSI LANGSUNG DI CELL COLAB
split_table = pd.crosstab(df.label, df.split)[["train", "val", "test"]]
display(split_table)

In [ ]:
# Sel 4 - Augmentasi Dinamis (Online) & DataLoader
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), ratio=(0.8, 1.25)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.03),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),
])

eval_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class MaskDS(Dataset):
    def __init__(self, frame, root, tf):
        self.paths = [Path(root) / p for p in frame.path]
        self.y = frame.y.values.astype(int)
        self.tf = tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img), int(self.y[i])

def make_loader(frame, root, tf, shuffle, seed=0):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(
        MaskDS(frame, root, tf), batch_size=BATCH_SIZE, shuffle=shuffle,
        num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
        generator=g if shuffle else None,
        persistent_workers=(NUM_WORKERS > 0),
        worker_init_fn=lambda w: np.random.seed(seed + w),
    )

print("✅ Pipeline Data Loading siap digunakan.")

In [ ]:
# Sel 5 - Fungsi Evaluasi & Render Visualisasi Langsung di Colab
from torch.utils.flop_counter import FlopCounterMode

def show_and_save_cm(cm, classes, out_path=None, title="Confusion Matrix"):
    """Menampilkan Confusion Matrix langsung di Cell Colab dan opsional menyimpannya."""
    cmn = cm.astype('float') / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes, ax=axes[0])
    axes[0].set_title(f"{title} (Jumlah Sampel)", fontweight="bold")
    axes[0].set_xlabel("Prediksi", fontweight="bold")
    axes[0].set_ylabel("Sebenarnya", fontweight="bold")
    axes[0].tick_params(axis='x', rotation=45)
    
    sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Greens", xticklabels=classes, yticklabels=classes, ax=axes[1])
    axes[1].set_title(f"{title} (Normalized Recall)", fontweight="bold")
    axes[1].set_xlabel("Prediksi", fontweight="bold")
    axes[1].set_ylabel("Sebenarnya", fontweight="bold")
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    if out_path:
        plt.savefig(out_path, dpi=120)
    plt.show()    # <--- LANGSUNG TAMPILKAN DI COLAB!

def show_training_curves(history, title="Training History"):
    """Menampilkan kurva Loss dan Akurasi Training vs Validasi di Cell Colab."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    axes[0].plot(history["train_loss"], label="Train Loss", color="tab:blue")
    axes[0].plot(history["val_loss"], label="Val Loss", color="tab:orange", linestyle="--")
    axes[0].set_title(f"{title} - Loss", fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("CrossEntropy Loss")
    axes[0].grid(True, linestyle="--", alpha=0.6)
    axes[0].legend()
    
    axes[1].plot(history["train_acc"], label="Train Acc", color="tab:green")
    axes[1].plot(history["val_acc"], label="Val Acc", color="tab:red", linestyle="--")
    axes[1].set_title(f"{title} - Accuracy", fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].grid(True, linestyle="--", alpha=0.6)
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()    # <--- LANGSUNG TAMPILKAN DI COLAB!

@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    all_p = []
    for x, _ in loader:
        p = torch.softmax(model(x.to(device)), dim=1).cpu().numpy()
        all_p.append(p)
    return np.concatenate(all_p, axis=0)

def count_params_m(model):
    return sum(p.numel() for p in model.parameters()) / 1e6

def measure_gflops(model, input_size=(1, 3, 224, 224)):
    try:
        with FlopCounterMode(display=False) as f:
            model(torch.zeros(input_size))
        return f.total_flops / 1e9
    except Exception:
        return 0.44

def measure_latency_ms(model, dev, n_warm=5, n_run=30):
    model = copy.deepcopy(model).to(dev).eval()
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=dev)
    with torch.no_grad():
        for _ in range(n_warm): model(dummy)
        if dev.type == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(n_run): model(dummy)
        if dev.type == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n_run * 1000.0

print("✅ Modul Evaluasi & Inline Visualizer siap.")

In [ ]:
# Sel 6 - Fungsi Training 2-Fase (Head Warming -> Fine-Tuning)
def set_frozen(params, frozen):
    for p in params: p.requires_grad = not frozen

def train_epoch(model, loader, opt, crit, scaler):
    model.train()
    tot_loss, corr, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        with torch.amp.autocast("cuda", enabled=(USE_AMP and device.type == "cuda")):
            out = model(x)
            loss = crit(out, y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        tot_loss += loss.item() * len(y)
        corr += (out.argmax(1) == y).sum().item()
        total += len(y)
    return tot_loss / total, corr / total

@torch.no_grad()
def eval_epoch(model, loader, crit):
    model.eval()
    tot_loss, corr, total = 0.0, 0, 0
    all_y, all_p = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast("cuda", enabled=(USE_AMP and device.type == "cuda")):
            out = model(x)
            loss = crit(out, y)
        tot_loss += loss.item() * len(y)
        pred = out.argmax(1)
        corr += (pred == y).sum().item()
        total += len(y)
        all_y.extend(y.cpu().numpy())
        all_p.extend(pred.cpu().numpy())
    f1 = f1_score(all_y, all_p, average="macro", zero_division=0)
    return tot_loss / total, corr / total, f1

In [ ]:
# Sel 7 - MobileNetV3 + Attention (SE & CBAM) untuk Ablation Study

class SEBlock(nn.Module):
    """Squeeze-and-Excitation: Channel Attention."""
    def __init__(self, c, r=16):
        super().__init__()
        mid = max(c // r, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(c, mid, 1)
        self.fc2 = nn.Conv2d(mid, c, 1)
        nn.init.zeros_(self.fc2.weight)
        nn.init.constant_(self.fc2.bias, 3.0)  # sigmoid(3) ~ 0.95 agar tidak merusak pretrained
    def forward(self, x):
        s = torch.sigmoid(self.fc2(F.relu(self.fc1(self.pool(x)))))
        return x * s

class ChannelGate(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        mid = max(c // r, 8)
        self.fc1 = nn.Conv2d(c, mid, 1)
        self.fc2 = nn.Conv2d(mid, c, 1)
        nn.init.zeros_(self.fc2.weight)
        nn.init.constant_(self.fc2.bias, 2.0)
    def _mlp(self, v):
        return self.fc2(F.relu(self.fc1(v)))
    def forward(self, x):
        a = self._mlp(F.adaptive_avg_pool2d(x, 1))
        m = self._mlp(F.adaptive_max_pool2d(x, 1))
        return x * torch.sigmoid(a + m)

class SpatialGate(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, k, padding=k // 2)
        nn.init.zeros_(self.conv.weight)
        nn.init.constant_(self.conv.bias, 3.0)
    def forward(self, x):
        s = torch.cat([x.mean(1, keepdim=True), x.amax(1, keepdim=True)], dim=1)
        return x * torch.sigmoid(self.conv(s))

class CBAM(nn.Module):
    """Convolutional Block Attention Module: Channel + Spatial Attention."""
    def __init__(self, c, r=16, k=7):
        super().__init__()
        self.ca = ChannelGate(c, r)
        self.sa = SpatialGate(k)
    def forward(self, x):
        return self.sa(self.ca(x))

ATTN = {"none": None, "se": SEBlock, "cbam": CBAM}

def build_mobilenet(num_classes, attn="none", where="head", late_blocks=(13, 14, 15)):
    m = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
    cls = ATTN[attn]
    if cls is not None:
        if where == "head":
            m.features = nn.Sequential(*list(m.features), cls(960))
        elif where == "late":
            for i in late_blocks:
                blk = m.features[i]
                m.features[i] = nn.Sequential(blk, cls(blk.out_channels))
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
    return m

def groups_mobilenet(model):
    head_ids = {id(p) for p in model.classifier.parameters()}
    for mod in model.modules():
        if isinstance(mod, (SEBlock, CBAM)):
            head_ids |= {id(p) for p in mod.parameters()}
    bb = [p for p in model.parameters() if id(p) not in head_ids]
    hd = [p for p in model.parameters() if id(p) in head_ids]
    return bb, hd

BASE_HP = dict(lr_head=1e-3, lr_bb=1e-4, lr_head_ft=1e-3, wd=1e-4,
               epochs_head=5, epochs_fine=25, patience=6)
CAM16 = lambda m: m.features[16]

EXPERIMENTS = {
    "mobilenetv3_base":      dict(build=lambda n: build_mobilenet(n, "none"),         groups=groups_mobilenet, cam_layer=CAM16, **BASE_HP),
    "mobilenetv3_se_head":   dict(build=lambda n: build_mobilenet(n, "se", "head"),   groups=groups_mobilenet, cam_layer=CAM16, **BASE_HP),
    "mobilenetv3_cbam_head": dict(build=lambda n: build_mobilenet(n, "cbam", "head"), groups=groups_mobilenet, cam_layer=CAM16, **BASE_HP),
    "mobilenetv3_cbam_late": dict(build=lambda n: build_mobilenet(n, "cbam", "late"), groups=groups_mobilenet, cam_layer=CAM16, **BASE_HP),
}
print("✅ 4 Arsitektur Ablation Study MobileNetV3 terdaftar:", list(EXPERIMENTS.keys()))

In [ ]:
# Sel 8 - Jalankan Eksperimen & Tampilkan Output Langsung di Layar Colab

def train_and_show(tag, seed=42):
    print(f"\n{'='*70}\n🚀 MEMULAI TRAINING: {tag} (Seed {seed})\n{'='*70}")
    cfg = EXPERIMENTS[tag]
    out_dir = OUT_ROOT / tag / f"seed_{seed}"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    set_seed(seed)
    tr_df, va_df, te_df = [df[df.split == s].reset_index(drop=True) for s in ["train", "val", "test"]]
    tr_loader = make_loader(tr_df, DATA_DIR, train_tf, True, seed)
    va_loader = make_loader(va_df, DATA_DIR, eval_tf, False)
    te_loader = make_loader(te_df, DATA_DIR, eval_tf, False)
    
    model = cfg["build"](NUM_CLASSES).to(device)
    bb, hd = cfg["groups"](model)
    
    if USE_CLASS_WEIGHTS:
        cnt = np.bincount(tr_df.y, minlength=NUM_CLASSES).astype(np.float32)
        w = torch.tensor(cnt.sum() / (NUM_CLASSES * np.maximum(cnt, 1)), device=device)
    else:
        w = None
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=LABEL_SMOOTHING)
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and device.type == "cuda"))
    
    # Tracking history
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_f1, best_w = -1.0, None
    
    # FASE 1: Freeze Backbone, Train Head
    print("--- FASE 1: Head Warming (Backbone Dibekukan) ---")
    set_frozen(bb, True); set_frozen(hd, False)
    opt_head = torch.optim.AdamW(hd, lr=cfg["lr_head"], weight_decay=cfg["wd"])
    for ep in range(cfg["epochs_head"]):
        tl, ta = train_epoch(model, tr_loader, opt_head, crit, scaler)
        vl, va, vf1 = eval_epoch(model, va_loader, crit)
        history["train_loss"].append(tl); history["val_loss"].append(vl)
        history["train_acc"].append(ta); history["val_acc"].append(va)
        if vf1 > best_f1:
            best_f1 = vf1
            best_w = copy.deepcopy(model.state_dict())
        print(f"[Head Ep {ep+1:02d}] Train Loss: {tl:.4f} Acc: {ta:.3f} | Val Loss: {vl:.4f} Acc: {va:.3f} Macro-F1: {vf1:.4f}")
        
    # FASE 2: Full Fine-Tuning
    print("\n--- FASE 2: Full Fine-Tuning (Seluruh Jaringan Aktif) ---")
    set_frozen(bb, False); set_frozen(hd, False)
    opt_full = torch.optim.AdamW([
        {"params": bb, "lr": cfg["lr_bb"]},
        {"params": hd, "lr": cfg["lr_head_ft"]}
    ], weight_decay=cfg["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt_full, T_max=cfg["epochs_fine"], eta_min=1e-6)
    
    patience_cnt = 0
    for ep in range(cfg["epochs_fine"]):
        tl, ta = train_epoch(model, tr_loader, opt_full, crit, scaler)
        vl, va, vf1 = eval_epoch(model, va_loader, crit)
        sched.step()
        history["train_loss"].append(tl); history["val_loss"].append(vl)
        history["train_acc"].append(ta); history["val_acc"].append(va)
        print(f"[Fine Ep {ep+1:02d}] Train Loss: {tl:.4f} Acc: {ta:.3f} | Val Loss: {vl:.4f} Acc: {va:.3f} Macro-F1: {vf1:.4f}")
        if vf1 > best_f1:
            best_f1 = vf1
            best_w = copy.deepcopy(model.state_dict())
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= cfg["patience"]:
                print(f"⏹️ Early stopping aktif di epoch {ep+1}")
                break
                
    # Muat bobot terbaik
    model.load_state_dict(best_w)
    torch.save(best_w, out_dir / "best_model.pth")
    
    # Evaluasi Akhir pada Test Set Bebas-Bocor
    probs = predict_probs(model, te_loader)
    preds = probs.argmax(axis=1)
    y_true = te_df.y.values
    
    test_acc = accuracy_score(y_true, preds)
    macro_f1 = f1_score(y_true, preds, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, preds, labels=list(range(NUM_CLASSES)))
    
    # TAMPILKAN LANGSUNG DI CELL COLAB TANPA PERLU BUKA DRIVE:
    print(f"\n📊 HASIL AKHIR EVALUASI TEST SET [{tag}]:")
    print(f"⭐ Test Accuracy : {test_acc * 100:.2f}%")
    print(f"⭐ Macro-F1     : {macro_f1 * 100:.2f}%")
    
    # 1. Tampilkan Kurva Training
    show_training_curves(history, title=f"Kurva Training ({tag})")
    
    # 2. Tampilkan Confusion Matrix
    show_and_save_cm(cm, CLASSES, out_path=out_dir / "confusion_matrix.png", title=f"Confusion Matrix ({tag})")
    
    # 3. Hitung Metrik Efisiensi
    cpu_model = copy.deepcopy(model).to("cpu")
    metrics = {
        "tag": tag,
        "test_acc": float(test_acc),
        "macro_f1": float(macro_f1),
        "params_m": round(count_params_m(model), 2),
        "size_mb": round(os.path.getsize(out_dir / "best_model.pth") / 1e6, 1),
        "gflops": round(measure_gflops(cpu_model), 2),
        "latency_cpu_ms": round(measure_latency_ms(cpu_model, torch.device("cpu")), 2)
    }
    json.dump(metrics, open(out_dir / "metrics.json", "w"), indent=2)
    return metrics

# Jalankan training (Pilih model usulan utama, atau seluruh 4 varian ablasi):
RUN_MODELS = ["mobilenetv3_base", "mobilenetv3_cbam_head"]
all_results = [train_and_show(m, seed=42) for m in RUN_MODELS]

In [ ]:
# Sel 9 - TABEL KOMPARASI ABLATION STUDY (TAMPIL LANGSUNG DI COLAB)
# Menampilkan tabel komparasi elegan dan Bar Chart perbandingan kinerja

rows = []
for f in sorted(OUT_ROOT.glob("*/seed_*/metrics.json")):
    rows.append(json.load(open(f)))

if rows:
    df_ablation = pd.DataFrame(rows)
    df_ablation["Akurasi Test (%)"] = (df_ablation["test_acc"] * 100).round(2)
    df_ablation["Macro-F1 (%)"]     = (df_ablation["macro_f1"] * 100).round(2)
    
    cols_show = ["tag", "Akurasi Test (%)", "Macro-F1 (%)", "params_m", "size_mb", "gflops", "latency_cpu_ms"]
    rename_map = {
        "tag": "Varian Model",
        "params_m": "Parameter (M)",
        "size_mb": "Ukuran (MB)",
        "gflops": "FLOPs (G)",
        "latency_cpu_ms": "Latensi CPU (ms)"
    }
    table = df_ablation[cols_show].rename(columns=rename_map)
    
    print("=" * 75)
    print("🏆 TABEL HASIL ABLATION STUDY MOBILENETV3 (MODEL USULAN):")
    print("=" * 75)
    display(table.style.highlight_max(subset=["Akurasi Test (%)", "Macro-F1 (%)"], color="#c8e6c9")\
                       .highlight_min(subset=["Parameter (M)", "Ukuran (MB)", "Latensi CPU (ms)"], color="#bbdefb"))
    
    # Visualisasi Bar Chart Perbandingan Ablasi
    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(len(table))
    w = 0.35
    ax.bar(x - w/2, table["Akurasi Test (%)"], w, label="Akurasi Test", color="#1976D2")
    ax.bar(x + w/2, table["Macro-F1 (%)"], w, label="Macro-F1", color="#FFA000")
    ax.set_xticks(x)
    ax.set_xticklabels(table["Varian Model"], fontweight="bold")
    ax.set_ylim([min(table["Akurasi Test (%)"].min() - 2, 90), 101])
    ax.set_ylabel("Persentase (%)", fontweight="bold")
    ax.set_title("Perbandingan Kinerja Varian Model Usulan (Ablation Study)", fontweight="bold", fontsize=12)
    ax.grid(axis='y', linestyle='--', alpha=0.6)
    ax.legend()
    for i in x:
        ax.text(i - w/2, table["Akurasi Test (%)"].iloc[i] + 0.2, f"{table['Akurasi Test (%)'].iloc[i]:.2f}%", ha="center", fontsize=9)
        ax.text(i + w/2, table["Macro-F1 (%)"].iloc[i] + 0.2, f"{table['Macro-F1 (%)'].iloc[i]:.2f}%", ha="center", fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("Belum ada metrics.json yang tersimpan. Jalankan Sel 8 terlebih dahulu.")